# Reading Vector Data


In the previous section, we explored spatial data models, looked at the principles used to describe them, and created a spatial dataset as a **GeoDataFrame** from point coordinates.

In practice, however, spatial analysis usually relies on datasets that have already been prepared and saved as files, which must then be loaded and processed correctly.


> In this section, we will look at the formats used to store spatial data and how to load them in Python.


## 0. Importing Libraries


In [ ]:
import pandas as pd
import geopandas as gpd

- [**pandas**](https://pandas.pydata.org/) (`pandas`) is a Python library for working with tabular data.

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) is a Python library that extends pandas to support geospatial data. With it you can load, process and analyse spatial datasets in a range of formats.


## 1. Vector Data Formats


Several specialised file formats exist for storing spatial vector data. The most common formats include:

1. **Shapefile (.shp)**

   A format developed by Esri and widely used in GIS applications. It consists of a set of interrelated files (`.shp` – geometry, `.shx` – index, `.dbf` – attributes) that together form a single dataset.

2. **GeoJSON (.geojson)**

   A text-based format built on JSON, widely used in web mapping. It is convenient for data exchange and easy to read, but not particularly efficient for large datasets.

3. **GeoPackage (.gpkg)**

   A modern spatial data storage format based on SQLite. It is an open standard developed by the Open Geospatial Consortium and is essentially a self-contained spatial database.


Spatial data is loaded using the `read_file()` function from the **GeoPandas** library.

The examples below use four files from `data/vienna/`:

- **vienna_playgrounds.shp** – public playgrounds in Vienna;
- **vienna_metro.geojson** – U-Bahn stations;
- **vienna_admin.gpkg** – boundaries of the city's districts and census districts;
- **vienna_top_locations.csv** – well-known places to visit in Vienna.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._

The examples below demonstrate how to import data in each of these formats.

You can apply the same commands to your own data in equivalent formats.


### 1.1. Shapefile


When working with this format, keep in mind that it is not a single file but a collection of interrelated files. The minimum set required for correct reading is:

- `.shp` – geometry
- `.dbf` – attributes
- `.shx` – index
- `.prj` – coordinate reference system

Although you only specify the path to the `.shp` file when loading, all the other files must be present in the same directory.


In [ ]:
gdf_shp = gpd.read_file("../../data/vienna/vienna_playgrounds_shp/vienna_playgrounds.shp")

The `head()` method shows the first rows of the attribute table:


In [ ]:
gdf_shp.head()

And `explore()` puts the data on an interactive map:

_The `tiles` parameter controls the background map. This example uses the light `cartodbpositron` basemap. If the parameter is omitted, the default **OpenStreetMap** basemap is used._


In [ ]:
gdf_shp.explore(tiles="cartodbpositron")

### 1.2. GeoJSON


Unlike the Shapefile format, **GeoJSON** is a **single file** that stores geometry, attributes, and CRS information together in a plain-text format.


In [ ]:
gdf_geojson = gpd.read_file("../../data/vienna/vienna_metro.geojson")

The first rows of the attribute table:


In [ ]:
gdf_geojson.head()

And the same data on a map:


In [ ]:
gdf_geojson.explore(tiles="cartodbpositron")

### 1.3. GeoPackage


When working with `.gpkg` files, note that a single file can contain multiple datasets (layers). In that case, you must explicitly specify which layer to load; otherwise, the first available layer will be read by default.


To find out which layers a `.gpkg` file contains, use the `list_layers()` function.


In [ ]:
gpd.list_layers("../../data/vienna/vienna_admin.gpkg")

This returns a list of available layers; you can then pass the desired layer name to the `layer` parameter when reading the file.


In [ ]:
gdf_gpkg = gpd.read_file("../../data/vienna/vienna_admin.gpkg", layer="zaehlbezirk")

The first rows:


In [ ]:
gdf_gpkg.head()

And on the map:


In [ ]:
gdf_gpkg.explore(tiles="cartodbpositron")

## 2. Tabular Data


Sometimes spatial information is stored purely as coordinates, without any explicit geometry. For example, a tabular dataset may include longitude and latitude columns for each feature.
In this form, the data is not yet represented as spatial features. However, geometry can be constructed from those coordinates and the result stored as a **GeoDataFrame**.

One of the most common formats for tabular data is **CSV (Comma-Separated Values)** – a plain text format where values in each row are separated by a delimiter (usually a comma).

Here is how to build a spatial dataset from a table that carries coordinates.


### 2.1. Reading a CSV File


Load the CSV file using the `read_csv()` function from the **pandas** library.

This file does not follow the defaults `read_csv()` assumes. Its values are separated by semicolons rather than commas, and its decimal separator is a comma rather than a full stop – a convention common across continental Europe. Both have to be declared, or the coordinates arrive as unusable text:


In [ ]:
df_csv = pd.read_csv("../../data/vienna/vienna_top_locations.csv", sep=";", decimal=",")

The first rows show the structure of the table and tell us which columns hold the coordinates:


In [ ]:
df_csv.head()

The coordinate values are stored in the `geo_latitude` (Y) and `geo_longitude` (X) columns.

Store the column names in variables:


In [ ]:
X = "geo_longitude"
Y = "geo_latitude"

### 2.2. Creating a GeoDataFrame


Convert the coordinate columns into point geometry using the `points_from_xy()` function from GeoPandas:


In [ ]:
# drop rows with missing coordinates
df_csv = df_csv.dropna(subset=[X, Y]) 

# create the geometry
geometry = gpd.points_from_xy(df_csv[X], df_csv[Y])

# inspect the first five geometries
geometry[:5]

Create a `GeoDataFrame` by combining the original table with the generated geometry:


In [ ]:
gdf_csv = gpd.GeoDataFrame(df_csv, geometry=geometry, crs="EPSG:4326")

Here we:

- pass the original table `df_csv`,
- add the geometry,
- specify the coordinate reference system (`EPSG:4326` – the WGS-84 geographic CRS).


The first rows of the result:


In [ ]:
gdf_csv.head()

And the locations on a map:


In [ ]:
gdf_csv.explore(tiles="cartodbpositron")

### 2.3. Saving the Result


Once a `GeoDataFrame` has been created, the spatial dataset can be saved in any supported format using the `to_file()` method from **GeoPandas**.


In [ ]:
# save as Shapefile
# gdf_csv.to_file("gdf_csv.shp")

# save as GeoJSON
# gdf_csv.to_file("gdf_csv.geojson", driver="GeoJSON")

# save as GeoPackage
# gdf_csv.to_file("gdf_csv.gpkg", layer="data", driver="GPKG")

The output format is determined by the file extension and/or the `driver` parameter (optional). For `.gpkg` files, it is recommended to specify a layer name via the `layer` parameter.


## Summary


In this section, we covered the main formats used to store vector spatial data and learned how to import them into Python using the **GeoPandas** library.

We examined the characteristics of the **Shapefile**, **GeoJSON**, and **GeoPackage** formats, and explored the differences in their structure and loading behaviour.

We also learned how to work with tabular data in **CSV** format, construct geometry from coordinates, and build a **GeoDataFrame** object.
